In [ ]:
# ── Cell 1 ───────────────────────────────────────────────────────────
import os, sys
OUT_DIR = "ps5_tvm_reliability_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

class _Tee:
    def __init__(self, j, f): self._j, self._f, self._is_tee = j, f, True
    def write(self, d):
        try: self._j.write(d); self._j.flush()
        except: pass
        try: self._f.write(d); self._f.flush()
        except: pass
    def flush(self):
        try: self._j.flush()
        except: pass
    def __getattr__(self, n): return getattr(self._j, n)

if not getattr(sys.stdout, "_is_tee", False):
    _lf = open(os.path.join(OUT_DIR, "console.log"), "w", buffering=1, encoding="utf-8")
    sys.stdout = _Tee(sys.stdout, _lf)
    sys.stderr = _Tee(sys.stderr, _lf)
    print(f"[capture] -> {OUT_DIR}/console.log")

try:
    import matplotlib.pyplot as _plt
    if not getattr(_plt.show, "_wrapped", False):
        _orig = _plt.show
        def _show(*a, _c=[0], **k):
            for n in _plt.get_fignums():
                fig = _plt.figure(n)
                t = ((fig._suptitle.get_text().strip() if fig._suptitle else None)
                     or (fig.axes[0].get_title().strip() if fig.axes else None)
                     or f"fig_{_c[0]:02d}")
                s = "".join(c if c.isalnum() or c in "-_ " else "_" for c in t).strip().replace(" ","_")[:60]
                p = os.path.join(OUT_DIR, f"{s}_{_c[0]:02d}.png")
                try: fig.savefig(p, dpi=110, bbox_inches="tight"); print(f"[fig] {p}"); _c[0]+=1
                except Exception as e: print(f"[fig err] {e}")
            return _orig(*a, **k)
        _show._wrapped = True; _plt.show = _show
    print(f"[capture] matplotlib -> {OUT_DIR}/")
except Exception as e: print("[capture] matplotlib not ready:", e)


# PS5 — TVM Device Reliability Analysis
Survival analysis, fault risk ranking, reliability growth, and RUL for **Ticket Vending Machines**.

**Data sources**:
- `silver.device_incident_features_daily` (S24) — event-level incidents with fault codes (`event_code_id`)
- `gold.device_ps1_daily` — device-type mapping (`mars_device_category`); also used as TBF fallback via `days_since_last_incident`

**Sections**: KM Survival · Weibull · Cox PH · Risk Ranking · Severity · Fault Heatmap · Crow-AMSAA · RBD · RUL

In [ ]:
# ── Cell 2 ───────────────────────────────────────────────────────────
# ── Install ───────────────────────────────────────────────────────────────────
import subprocess, sys
pkgs = ["lifelines", "scikit-learn", "matplotlib", "seaborn",
        "pandas", "numpy", "pyarrow", "boto3", "s3fs", "scipy", "networkx", "mlxtend"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)
print("Install complete.")


In [ ]:
# ── Cell 3 ───────────────────────────────────────────────────────────
# ── Imports ───────────────────────────────────────────────────────────────────
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import networkx as nx

from lifelines import KaplanMeierFitter, WeibullFitter, NelsonAalenFitter, CoxPHFitter
from lifelines.statistics import logrank_test
from sklearn.preprocessing import StandardScaler
from scipy.stats import weibull_min

pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:.4f}".format)
sns.set_style("whitegrid")
print("Imports OK")


In [ ]:
# ── Cell 4 ───────────────────────────────────────────────────────────
# ── Configuration ─────────────────────────────────────────────────────────────
DEVICE_TYPE = "TVM"
REGION      = "us-east-1"

# Primary: device_event_enriched — raw device telemetry events logged by the device itself.
# These are hardware/software fault events, NOT ServiceNow incident tickets.
# incident_history = ServiceNow tickets (delayed, not all failures captured).
# device_event_enriched = every event the device reported (precise timing, full coverage).
SILVER_EVT_S3 = "s3://cubic-mars-pm-s3-datalake-dev-gold-170202974600/chicago/silver/device_event_enriched"

# Gold PS1 daily — device-type mapping + S24 features; used as fallback if silver unavailable
GOLD_PS1_S3 = "s3://cubic-mars-pm-s3-datalake-dev-gold-170202974600/chicago/gold/device_ps1_daily"

S3_OPTS    = {"client_kwargs": {"region_name": REGION}}
DEV_COL    = "mars_device_category"
DEV_KEY    = "DEVICE_KEY"
DEV_ID_COL = "DEVICE_ID"
FC_COL     = "event_code_id"   # device event code — present in device_event_enriched
DATE_COL   = "event_date"      # device_event_enriched uses event_date
TOP_N      = 10
RANDOM_STATE = 42

print(f"Device type : {DEVICE_TYPE}")
print(f"Silver path : {SILVER_EVT_S3}")
print(f"Gold path   : {GOLD_PS1_S3}")

In [ ]:
# ── Cell 5 ───────────────────────────────────────────────────────────
# ── Load Data ─────────────────────────────────────────────────────────────────
USING_GOLD_FALLBACK = False
TBF_COL_GOLD = "days_since_last_incident"

# Candidate paths — device_event_enriched first (raw device telemetry, NOT ServiceNow tickets).
# incident_history = ServiceNow tickets only; misses many device failures and has coarser timing.
_SILVER_CANDIDATES = [
    "s3://cubic-mars-pm-s3-datalake-dev-gold-170202974600/chicago/silver/device_event_enriched",
    "s3://cubic-mars-pm-s3-datalake-dev-silver-170202974600/chicago/silver/device_event_enriched",
    # incident_history as last-resort silver fallback (ServiceNow tickets)
    "s3://cubic-mars-pm-s3-datalake-dev-gold-170202974600/chicago/silver/incident_history",
]

# Reject tables that have no real event/fault code (only aggregate count columns)
_COUNT_ONLY_COLS = {"distinct_event_codes_30d"}

def _has_real_fault_code(cols):
    for c in cols:
        cl = c.lower()
        if ("code" in cl or "fault" in cl or "event_type" in cl) and c not in _COUNT_ONLY_COLS:
            return True
    return False

# Pre-load gold for dev_map — lets us filter silver at read time instead of post-load.
# Gold PS1 is device-level daily aggregates (small); silver is event-level (large).
print("Pre-loading device-type mapping from gold ...")
_gold_pre = pd.read_parquet(GOLD_PS1_S3, storage_options=S3_OPTS)
_map_cols_pre = [c for c in [DEV_KEY, DEV_ID_COL, DEV_COL, "DEVICE_NAME"] if c in _gold_pre.columns]
dev_map = _gold_pre[_map_cols_pre].drop_duplicates(subset=[DEV_KEY]).copy()
if DEV_COL in dev_map.columns:
    dev_map[DEV_COL] = dev_map[DEV_COL].astype(str).str.strip().str.upper()
    _dev_keys = dev_map[dev_map[DEV_COL] == DEVICE_TYPE][DEV_KEY].unique().tolist()
else:
    _dev_keys = None
del _gold_pre
print(f"  Pre-filter: {len(_dev_keys) if _dev_keys else '(all)'} {DEVICE_TYPE} device keys")

df_inc = None
SILVER_EVT_S3_USED = None
for _path in _SILVER_CANDIDATES:
    try:
        print(f"Trying: {_path} ...")
        _pq_filter = [(DEV_KEY, "in", _dev_keys)] if _dev_keys else None
        try:
            _tmp = pd.read_parquet(_path, storage_options=S3_OPTS, filters=_pq_filter)
        except Exception:
            _tmp = pd.read_parquet(_path, storage_options=S3_OPTS)
        print(f"  OK: {_tmp.shape[0]:,} rows x {_tmp.shape[1]} cols")
        print(f"  Columns: {_tmp.columns.tolist()}")
        if not _has_real_fault_code(_tmp.columns):
            print(f"  SKIP: no real fault/event code column — trying next path")
            continue
        df_inc = _tmp
        SILVER_EVT_S3_USED = _path
        break
    except Exception as _e:
        print(f"  FAIL: {str(_e)[:120]}")

if df_inc is not None:
    # Silver loaded — dev_map already built in pre-load above
    print(f"\nLoaded: {SILVER_EVT_S3_USED}")

    # Normalise DEVICE_KEY column name
    if DEV_KEY not in df_inc.columns:
        _dk_cands = [c for c in df_inc.columns if c.lower() == DEV_KEY.lower()]
        if _dk_cands:
            df_inc = df_inc.rename(columns={_dk_cands[0]: DEV_KEY})

    # Auto-detect date column (device_event_enriched uses event_date or event_datetime)
    if DATE_COL not in df_inc.columns:
        for _dc in ["event_date", "event_datetime", "incident_date", "transit_day", "created_at", "date"]:
            if _dc in df_inc.columns:
                DATE_COL = _dc
                print(f"  Date column auto-detected: {DATE_COL}")
                break

    df_inc[DATE_COL] = pd.to_datetime(df_inc[DATE_COL], errors="coerce")
    df = df_inc.merge(dev_map, on=DEV_KEY, how="left")
    if DEV_COL in df.columns:
        df = df[df[DEV_COL] == DEVICE_TYPE].copy()
        print(f"  Filtered to {DEVICE_TYPE}: {len(df):,} rows")

    # Confirm fault/event code column
    if FC_COL not in df.columns:
        _cands = [c for c in df.columns
                  if ("code" in c.lower() or "fault" in c.lower() or "event_type" in c.lower())
                  and c not in _COUNT_ONLY_COLS]
        FC_COL = _cands[0] if _cands else None
        if FC_COL:
            print(f"  Event code column: {FC_COL} (auto-detected)")
    else:
        print(f"  Event code column: {FC_COL}")
    if FC_COL:
        df[FC_COL] = df[FC_COL].astype(str)

    print(f"\n{DEVICE_TYPE} device events (silver): {len(df):,} rows")
    print(f"Unique event codes : {df[FC_COL].nunique():,}" if FC_COL else "[WARN] No event code column found")
    print(f"Unique devices     : {df[DEV_KEY].nunique():,}")
    if DATE_COL in df.columns and df[DATE_COL].notna().any():
        print(f"Date range         : {df[DATE_COL].min().date()} -> {df[DATE_COL].max().date()}")

else:
    # ── GOLD FALLBACK ─────────────────────────────────────────────────────────
    print("\n[GOLD FALLBACK] Silver device_event_enriched not accessible.")
    print("  Using gold PS1 daily — has real event_code_id + days_since_last_incident")
    USING_GOLD_FALLBACK = True

    df_gold_all = pd.read_parquet(GOLD_PS1_S3, storage_options=S3_OPTS)
    print(f"  Gold PS1: {df_gold_all.shape[0]:,} rows x {df_gold_all.shape[1]} cols")

    for _date_cand in ["transit_day", "event_date", "date", "incident_date"]:
        if _date_cand in df_gold_all.columns:
            DATE_COL = _date_cand
            break
    print(f"  Date column: {DATE_COL}")

    if DEV_COL in df_gold_all.columns:
        df_gold_all[DEV_COL] = df_gold_all[DEV_COL].astype(str).str.strip().str.upper()
        df = df_gold_all[df_gold_all[DEV_COL] == DEVICE_TYPE].copy()
    else:
        df = df_gold_all.copy()

    map_cols = [c for c in [DEV_KEY, DEV_ID_COL, DEV_COL] if c in df_gold_all.columns]
    dev_map = df_gold_all[map_cols].drop_duplicates(subset=[DEV_KEY]).copy()

    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
    del df_gold_all

    if TBF_COL_GOLD not in df.columns:
        _tbf_cands = [c for c in df.columns if "since" in c.lower() or "tbf" in c.lower()
                      or ("days" in c.lower() and "incident" in c.lower())]
        TBF_COL_GOLD = _tbf_cands[0] if _tbf_cands else None
        print(f"  [WARN] TBF column not found. Candidates: {_tbf_cands}")

    for _fc_cand in ["event_code_id", "fault_code", "event_type"]:
        if _fc_cand in df.columns:
            FC_COL = _fc_cand
            df[FC_COL] = df[FC_COL].astype(str)
            break

    print(f"\n{DEVICE_TYPE} rows (gold fallback): {len(df):,}")
    print(f"TBF column    : {TBF_COL_GOLD}")
    print(f"Event code    : {FC_COL}")
    print(f"Unique devices: {df[DEV_KEY].nunique():,}")
    if DATE_COL in df.columns and df[DATE_COL].notna().any():
        print(f"Date range    : {df[DATE_COL].min().date()} -> {df[DATE_COL].max().date()}")
    print("\n[NOTE] To use device telemetry events, export from Databricks:"
          "\n  spark.table('mars_dev.silver.device_event_enriched')"
          "\n    .write.mode('overwrite').parquet("
          "\n      's3://cubic-mars-pm-s3-datalake-dev-gold-170202974600/chicago/silver/device_event_enriched')"
          "\n  Then re-run from Cell 4.")

In [ ]:
# ── Cell 6 ───────────────────────────────────────────────────────────
# ── Compute TBF (Time Between Failures) per Device+FaultCode ─────────────────
if USING_GOLD_FALLBACK:
    # days_since_last_incident IS the TBF for rows where an incident occurred
    if TBF_COL_GOLD and TBF_COL_GOLD in df.columns:
        df_tbf = df[(df[TBF_COL_GOLD].notna()) & (df[TBF_COL_GOLD] > 0)].copy()
        df_tbf["TBF_days"] = df_tbf[TBF_COL_GOLD].astype(float)
        df_tbf["TBF_min"]  = df_tbf["TBF_days"] * 1440.0
        df_tbf["event"]    = 1
        df_sorted = df_tbf.copy()
        print(f"TBF (gold: days_since_last_incident): {len(df_tbf):,} rows")
    else:
        # No pre-computed TBF column — compute by date-diff per device
        df_sorted = df.sort_values([DEV_KEY, DATE_COL]).copy()
        df_sorted["_prev_date"] = df_sorted.groupby(DEV_KEY)[DATE_COL].shift(1)
        df_sorted["TBF_days"]   = (df_sorted[DATE_COL] - df_sorted["_prev_date"]).dt.days.astype(float)
        df_sorted["TBF_min"]    = df_sorted["TBF_days"] * 1440.0
        df_sorted["event"]      = 1
        df_tbf = df_sorted.dropna(subset=["TBF_days"]).copy()
        df_tbf = df_tbf[df_tbf["TBF_days"] > 0].copy()
        print(f"TBF (gold date-diff): {len(df_tbf):,} rows")
else:
    # Silver path — event-level; compute TBF per device x fault code
    _group = [DEV_KEY, FC_COL] if FC_COL else [DEV_KEY]
    df_sorted = df.sort_values(_group + [DATE_COL]).copy()
    df_sorted["_prev_date"] = df_sorted.groupby(_group)[DATE_COL].shift(1)
    df_sorted["TBF_min"]    = (df_sorted[DATE_COL] - df_sorted["_prev_date"]).dt.total_seconds() / 60.0
    df_sorted["TBF_days"]   = df_sorted["TBF_min"] / 1440.0
    df_sorted["event"]      = 1
    df_tbf = df_sorted.dropna(subset=["TBF_min"]).copy()
    df_tbf = df_tbf[df_tbf["TBF_min"] > 0].copy()
    print(f"TBF rows: {len(df_tbf):,}")

if len(df_tbf) > 0:
    print(df_tbf["TBF_days"].describe().round(2).to_string())

# ── Print top 10 fault codes and devices with failures ────────────────────────
top10_fc   = df_tbf[FC_COL].value_counts().head(10) if (FC_COL and FC_COL in df_tbf.columns) else pd.Series(dtype=int)
top10_devs = df_tbf[DEV_KEY].value_counts().head(10)

print(f"\n{'='*55}")
print(f"TOP 10 FAULT CODES -- {DEVICE_TYPE}")
print(f"{'='*55}")
# Event name lookup — check if silver has a description/name column alongside FC_COL
_name_cands = ([c for c in df.columns
                if any(k in c.lower() for k in ('name', 'desc', 'label', 'text'))
                and c != FC_COL and 'key' not in c.lower()]
               if FC_COL else [])
_evt_name_col = _name_cands[0] if _name_cands else None

if len(top10_fc) > 0:
    _fc_df = top10_fc.reset_index()
    _fc_df.columns = [FC_COL, 'failure_count']
    if _evt_name_col:
        _evt_map = (df[[FC_COL, _evt_name_col]].dropna(subset=[FC_COL])
                     .drop_duplicates(FC_COL).set_index(FC_COL)[_evt_name_col])
        _fc_df['event_name'] = _fc_df[FC_COL].map(_evt_map).fillna('(no description)')
        _fc_df = _fc_df[[FC_COL, 'event_name', 'failure_count']]
    else:
        print(f'  [NOTE] No event name column found in silver — {FC_COL} shows numeric codes.')
    print(_fc_df.to_string(index=False))
else:
    print('No event code data available.')
print(f"\n{'='*55}")
print(f"TOP 10 DEVICES BY FAILURE COUNT -- {DEVICE_TYPE}")
print(f"{'='*55}")
# Device ID lookup via dev_map
_dev_id_col = DEV_ID_COL if DEV_ID_COL in dev_map.columns else None
_id_map = (dev_map[[DEV_KEY, _dev_id_col]].drop_duplicates(DEV_KEY)
           .set_index(DEV_KEY)[_dev_id_col]) if _dev_id_col else None
_dev_df = top10_devs.reset_index()
_dev_df.columns = [DEV_KEY, 'failure_count']
if _id_map is not None:
    _dev_df.insert(0, _dev_id_col, _dev_df[DEV_KEY].map(_id_map))
print(_dev_df.to_string(index=False))

TOP10_FC = top10_fc.index.tolist() if len(top10_fc) > 0 else []
SELECTED_DEVICE = top10_devs.index[0] if len(top10_devs) > 0 else None
_sel_id = (_id_map.get(SELECTED_DEVICE, "unknown")
           if (_id_map is not None and SELECTED_DEVICE is not None) else None)
print(f"\nSelected device for per-device analysis: {SELECTED_DEVICE}"
      + (f"  ({_dev_id_col}: {_sel_id})" if _sel_id else ""))

In [ ]:
# ── Cell 6b — MTTR + Availability (ServiceNow incident_history) ──────────────
# MTTR = Mean Time To Repair from ServiceNow ticket open→resolve timestamps
# Availability = MTBF / (MTBF + MTTR)  — classic reliability triangle
# device_event_enriched → MTBF  |  incident_history → MTTR  |  combined → Availability

_SN_CANDIDATES = [
    "s3://cubic-mars-pm-s3-datalake-dev-gold-170202974600/chicago/silver/incident_history",
    "s3://cubic-mars-pm-s3-datalake-dev-silver-170202974600/chicago/silver/incident_history",
]

df_sn = None
for _path in _SN_CANDIDATES:
    try:
        print(f"Loading incident_history (ServiceNow): {_path} ...")
        _tmp = pd.read_parquet(_path, storage_options=S3_OPTS)
        print(f"  OK: {_tmp.shape[0]:,} rows  Cols: {_tmp.columns.tolist()}")
        df_sn = _tmp
        break
    except Exception as _e:
        print(f"  FAIL: {str(_e)[:120]}")

MTTR_DEV  = pd.Series(dtype=float)
MTTR_FC   = pd.Series(dtype=float)
AVAIL_DEV = pd.Series(dtype=float)

if df_sn is not None:
    # Normalise DEVICE_KEY
    if DEV_KEY not in df_sn.columns:
        _dk = [c for c in df_sn.columns if c.lower() == DEV_KEY.lower()]
        if _dk:
            df_sn = df_sn.rename(columns={_dk[0]: DEV_KEY})

    # Filter to this device type via dev_map
    if DEV_KEY in df_sn.columns and DEV_KEY in dev_map.columns:
        _sn_map_cols = [c for c in [DEV_KEY, DEV_COL] if c in dev_map.columns]
        df_sn = df_sn.merge(dev_map[_sn_map_cols].drop_duplicates(DEV_KEY),
                             on=DEV_KEY, how="left")
        if DEV_COL in df_sn.columns:
            df_sn[DEV_COL] = df_sn[DEV_COL].astype(str).str.strip().str.upper()
            df_sn = df_sn[df_sn[DEV_COL] == DEVICE_TYPE].copy()
            print(f"  Filtered to {DEVICE_TYPE}: {len(df_sn):,} incidents")

    # Auto-detect open / resolve timestamps and pre-computed TTR
    _open_cands  = ["opened_dtm", "opened_at", "open_dtm", "created_at",
                    "sys_created_on", "open_at", "reported_at"]
    _close_cands = ["resolved_dtm", "resolved_at", "closed_dtm", "closed_at",
                    "resolution_date", "sys_updated_on"]
    _ttr_cands   = ["time_to_resolve_minutes", "time_to_close_minutes",
                    "duration_seconds", "time_to_resolve", "resolve_time",
                    "ttr_minutes", "ttr_hours", "resolution_time_minutes"]

    _open_col  = next((c for c in _open_cands  if c in df_sn.columns), None)
    _close_col = next((c for c in _close_cands if c in df_sn.columns), None)
    _ttr_col   = next((c for c in _ttr_cands   if c in df_sn.columns), None)
    print(f"  Open col : {_open_col}  |  Close col : {_close_col}  |  Pre-calc TTR : {_ttr_col}")

    if _ttr_col:
        df_sn["TTR_hours"] = pd.to_numeric(df_sn[_ttr_col], errors="coerce")
        if "second" in _ttr_col.lower():
            df_sn["TTR_hours"] /= 3600.0
        elif "minute" in _ttr_col.lower():
            df_sn["TTR_hours"] /= 60.0
    elif _open_col and _close_col:
        df_sn[_open_col]  = pd.to_datetime(df_sn[_open_col],  errors="coerce")
        df_sn[_close_col] = pd.to_datetime(df_sn[_close_col], errors="coerce")
        df_sn["TTR_hours"] = (df_sn[_close_col] - df_sn[_open_col]).dt.total_seconds() / 3600.0
    else:
        print("  [WARN] No TTR timestamps or pre-computed column found — MTTR unavailable")
        df_sn = None

if df_sn is not None and "TTR_hours" in df_sn.columns:
    # Sanity filter: exclude negatives and implausible values (> 1 year)
    df_sn = df_sn[(df_sn["TTR_hours"] > 0) & (df_sn["TTR_hours"] < 8760)].copy()

    # MTTR per device
    MTTR_DEV = df_sn.groupby(DEV_KEY)["TTR_hours"].mean()

    # MTTR per fault code
    _fc_sn = FC_COL if FC_COL in df_sn.columns else next(
        (c for c in df_sn.columns if "code" in c.lower() and c not in _COUNT_ONLY_COLS), None)
    if _fc_sn:
        MTTR_FC = df_sn.groupby(_fc_sn)["TTR_hours"].mean()

    print(f"\nMTTR Summary — {DEVICE_TYPE} ({len(df_sn):,} incidents, {MTTR_DEV.index.nunique()} devices):")
    print(f"  Mean MTTR   : {MTTR_DEV.mean():.2f} h")
    print(f"  Median MTTR : {MTTR_DEV.median():.2f} h")
    print(f"  Max MTTR    : {MTTR_DEV.max():.2f} h")

    # ── Availability = MTBF / (MTBF + MTTR) ──────────────────────────────────
    # Use per-device MTBF derived from TBF data already computed in Cell 6
    _mtbf_dev = df_tbf.groupby(DEV_KEY)["TBF_days"].mean().rename("MTBF_days")
    avail_df = _mtbf_dev.to_frame().join(
        (MTTR_DEV / 24.0).rename("MTTR_days"), how="inner"
    ).reset_index()
    avail_df["Availability_%"] = (
        avail_df["MTBF_days"] / (avail_df["MTBF_days"] + avail_df["MTTR_days"])
    ) * 100
    AVAIL_DEV = avail_df.set_index(DEV_KEY)["Availability_%"]

    print(f"\nAvailability = MTBF / (MTBF + MTTR)  [{len(avail_df)} devices with both metrics]")
    print(f"  Fleet mean        : {AVAIL_DEV.mean():.2f}%")
    print(f"  Fleet median      : {AVAIL_DEV.median():.2f}%")
    print(f"  Devices < 99%     : {(AVAIL_DEV < 99).sum()}")
    print(f"  Devices < 95%     : {(AVAIL_DEV < 95).sum()}")

    print(f"\nBottom 10 devices by Availability:")
    _avail_disp = avail_df.nsmallest(10, "Availability_%").copy()
    if _id_map is not None:
        _avail_disp.insert(0, DEV_ID_COL, _avail_disp[DEV_KEY].map(_id_map))
        _avail_cols = [DEV_ID_COL, DEV_KEY, "MTBF_days", "MTTR_days", "Availability_%"]
    else:
        _avail_cols = [DEV_KEY, "MTBF_days", "MTTR_days", "Availability_%"]
    print(_avail_disp[_avail_cols].round(2).to_string(index=False))

    # ── Plots ─────────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(22, 7))
    fig.suptitle(f"{DEVICE_TYPE} — MTTR + Availability Analysis (ServiceNow + Device Events)",
                 fontsize=13, fontweight="bold")

    # TTR distribution
    _ttr_clip = df_sn["TTR_hours"].clip(upper=df_sn["TTR_hours"].quantile(0.99))
    axes[0].hist(_ttr_clip, bins=50, color="#7B1FA2", alpha=0.8, edgecolor="white")
    axes[0].axvline(df_sn["TTR_hours"].median(), color="red", ls="--", lw=1.5,
                    label=f"Median={df_sn['TTR_hours'].median():.1f}h")
    axes[0].set(title="Time To Repair (TTR) Distribution",
                xlabel="TTR (hours)", ylabel="Count")
    axes[0].legend()

    # Top 10 worst MTTR devices
    _top_mttr = MTTR_DEV.nlargest(10)
    axes[1].barh(range(len(_top_mttr)), _top_mttr.values, color="#E53935", alpha=0.85)
    axes[1].set_yticks(range(len(_top_mttr)))
    _mttr_labels = ([str(_id_map.get(d, d)) for d in _top_mttr.index]
                    if _id_map is not None else [str(d) for d in _top_mttr.index])
    axes[1].set_yticklabels(_mttr_labels, fontsize=9)
    axes[1].set(title="Top 10 Devices — Worst MTTR (slowest to repair)",
                xlabel="MTTR (hours)")

    # Availability distribution
    if len(AVAIL_DEV) > 0:
        axes[2].hist(AVAIL_DEV.dropna(), bins=40, color="#1565C0", alpha=0.8, edgecolor="white")
        axes[2].axvline(AVAIL_DEV.mean(), color="red", ls="--", lw=1.5,
                        label=f"Mean={AVAIL_DEV.mean():.1f}%")
        axes[2].axvline(99.0, color="#2E7D32", ls=":", lw=1.5, label="99% target")
        axes[2].set(title="Device Availability Distribution",
                    xlabel="Availability (%)", ylabel="Devices")
        axes[2].legend()

    plt.tight_layout()
    plt.show()

    # MTTR per fault code bar chart
    if len(MTTR_FC) > 0:
        _top_mttr_fc = MTTR_FC.nlargest(10)
        fig2, ax2 = plt.subplots(figsize=(14, 6))
        fig2.suptitle(f"{DEVICE_TYPE} — MTTR per Fault Code (Top 10 slowest to repair)",
                      fontsize=13, fontweight="bold")
        ax2.barh(range(len(_top_mttr_fc)), _top_mttr_fc.values, color="#7B1FA2", alpha=0.85)
        ax2.set_yticks(range(len(_top_mttr_fc)))
        ax2.set_yticklabels([f"FC={fc}" for fc in _top_mttr_fc.index], fontsize=9)
        ax2.set(title="Mean Time To Repair per Fault Code",
                xlabel="MTTR (hours)")
        plt.tight_layout()
        plt.show()

    # MTBF vs MTTR scatter — quadrant view
    if len(avail_df) > 0:
        fig3, ax3 = plt.subplots(figsize=(12, 7))
        fig3.suptitle(f"{DEVICE_TYPE} — MTBF vs MTTR per Device (Reliability Quadrant)",
                      fontsize=13, fontweight="bold")
        sc = ax3.scatter(avail_df["MTBF_days"], avail_df["MTTR_days"],
                         c=avail_df["Availability_%"], cmap="RdYlGn",
                         vmin=90, vmax=100, s=40, alpha=0.7, edgecolors="none")
        plt.colorbar(sc, ax=ax3, label="Availability (%)")
        ax3.axvline(avail_df["MTBF_days"].median(), color="gray", ls="--", lw=1, alpha=0.6)
        ax3.axhline(avail_df["MTTR_days"].median(), color="gray", ls="--", lw=1, alpha=0.6)
        ax3.set(title="High MTBF + Low MTTR → Top-right = Best devices",
                xlabel="MTBF (days)", ylabel="MTTR (days)")
        plt.tight_layout()
        plt.show()

else:
    print(f"[SKIP] MTTR: incident_history not loaded or no TTR data found.")
    print("  Ensure incident_history is exported to S3 with resolved_at/closed_at columns.")

In [ ]:
# ── Cell 6c — Repeat Failure / Chronic Repeater Analysis ─────────────────────
# A repeat failure = same fault code on same device recurring within REPEAT_WINDOW days.
# High repeat rate = repair did not hold -> wrong root cause fix or inadequate PM.
REPEAT_WINDOW = 30  # days

if len(df_tbf) > 0 and FC_COL:
    df_repeat = df_tbf[df_tbf["TBF_days"] <= REPEAT_WINDOW].copy()

    fc_counts  = df_tbf[FC_COL].value_counts()
    fc_repeats = df_repeat[FC_COL].value_counts()
    repeat_rate = (fc_repeats / fc_counts * 100).dropna().sort_values(ascending=False)

    dev_repeat_count = df_repeat.groupby(DEV_KEY).size().sort_values(ascending=False)
    dev_total        = df_tbf.groupby(DEV_KEY).size()
    dev_repeat       = df_repeat.groupby(DEV_KEY).size()

    print("=" * 60)
    print(f"REPEAT FAILURE ANALYSIS - {DEVICE_TYPE}  (window: {REPEAT_WINDOW}d)")
    print("=" * 60)
    print(f"Total TBF intervals : {len(df_tbf):,}")
    print(f"Repeat failures     : {len(df_repeat):,}  ({len(df_repeat)/len(df_tbf)*100:.1f}%)")
    print(f"\nTop 10 Fault Codes by Repeat Rate (% recur within {REPEAT_WINDOW}d):")
    print(repeat_rate.head(10).round(1).to_string())
    print("\nTop 10 Chronic Repeater Devices:")
    print(dev_repeat_count.head(10).to_string())

    fig, axes = plt.subplots(1, 3, figsize=(22, 7))
    fig.suptitle(
        f"{DEVICE_TYPE} - Repeat Failure / Chronic Repeater Analysis (window={REPEAT_WINDOW}d)",
        fontsize=13, fontweight="bold"
    )

    _top_rr = repeat_rate.head(10)
    axes[0].barh(range(len(_top_rr)), _top_rr.values, color="#E53935", alpha=0.85)
    axes[0].set_yticks(range(len(_top_rr)))
    axes[0].set_yticklabels([f"FC={fc}" for fc in _top_rr.index], fontsize=9)
    axes[0].axvline(repeat_rate.mean(), color="black", ls="--", lw=1,
                    label=f"Fleet avg={repeat_rate.mean():.1f}%")
    axes[0].set(title=f"Repeat Rate per Fault Code (recur <={REPEAT_WINDOW}d)", xlabel="Repeat Rate (%)")
    axes[0].legend(fontsize=8)

    _top_dev_r = dev_repeat_count.head(10)
    axes[1].barh(range(len(_top_dev_r)), _top_dev_r.values, color="#FF6F00", alpha=0.85)
    axes[1].set_yticks(range(len(_top_dev_r)))
    axes[1].set_yticklabels([str(d) for d in _top_dev_r.index], fontsize=9)
    axes[1].set(title="Top 10 Chronic Repeater Devices", xlabel="Repeat Failure Count")

    _tbf_clip = df_tbf["TBF_days"].clip(upper=df_tbf["TBF_days"].quantile(0.99))
    axes[2].hist(_tbf_clip, bins=60, color="#1565C0", alpha=0.6, edgecolor="white", label="All TBF")
    axes[2].hist(df_repeat["TBF_days"].clip(upper=REPEAT_WINDOW), bins=30,
                 color="#E53935", alpha=0.8, edgecolor="white", label=f"Repeats (<={REPEAT_WINDOW}d)")
    axes[2].axvline(REPEAT_WINDOW, color="black", ls="--", lw=1.5, label=f"{REPEAT_WINDOW}d threshold")
    axes[2].set(title="TBF Distribution - Repeat vs Normal", xlabel="TBF (days)", ylabel="Count")
    axes[2].legend(fontsize=8)

    plt.tight_layout()
    plt.show()

    repeat_summary = pd.DataFrame({
        "total_failures"  : fc_counts,
        "repeat_failures" : fc_repeats.reindex(fc_counts.index, fill_value=0),
        "repeat_rate_pct" : repeat_rate.reindex(fc_counts.index, fill_value=0).round(1),
    }).sort_values("repeat_failures", ascending=False).head(15)
    print("\nRepeat Failure Summary - Top 15 by Repeat Count:")
    print(repeat_summary.to_string())
else:
    print("[SKIP] Repeat failure analysis - no TBF or fault code data.")


In [ ]:
# ── Cell 6d — Failure Precursor / Leading Indicator Analysis ─────────────────
# Finds event codes that reliably PRECEDE the top critical fault codes within
# a look-back window — early-warning signals for predictive alerting.
PRECURSOR_WINDOW_DAYS = 7
TOP_CRITICAL_FC = TOP10_FC[:5]  # top 5 most-frequent fault codes treated as critical
PRECURSOR_MAX_ROWS = 300_000   # cap df rows fed into cross-join to prevent OOM

if len(df) > 0 and FC_COL and DATE_COL in df.columns and len(TOP_CRITICAL_FC) > 0:
    precursor_results = {}

    for crit_fc in TOP_CRITICAL_FC:
        crit_events = df[df[FC_COL] == crit_fc][[DEV_KEY, DATE_COL]].copy()
        crit_events = crit_events.rename(columns={DATE_COL: "_crit_date"})
        if len(crit_events) < 5:
            continue

        # Sample df if too large — cross-join can spike 4-6x input size in RAM
        _df_sample = (df.sample(n=PRECURSOR_MAX_ROWS, random_state=42)
                      if len(df) > PRECURSOR_MAX_ROWS else df)
        merged = _df_sample.merge(crit_events, on=DEV_KEY, how="inner")
        merged["_days_before"] = (merged["_crit_date"] - merged[DATE_COL]).dt.days
        precursors = merged[
            (merged["_days_before"] > 0) &
            (merged["_days_before"] <= PRECURSOR_WINDOW_DAYS) &
            (merged[FC_COL] != crit_fc)
        ]
        if len(precursors) == 0:
            continue

        prec_freq    = precursors[FC_COL].value_counts()
        prec_support = (prec_freq / len(crit_events) * 100).round(1)
        precursor_results[crit_fc] = prec_support.head(5)

    if precursor_results:
        print("=" * 65)
        print(f"FAILURE PRECURSOR ANALYSIS - {DEVICE_TYPE}  (window={PRECURSOR_WINDOW_DAYS}d)")
        print("=" * 65)
        for crit_fc, prec in precursor_results.items():
            print(f"\nCritical FC={crit_fc} - precursors (% preceded by within {PRECURSOR_WINDOW_DAYS}d):")
            for p_fc, pct in prec.items():
                print(f"  FC={p_fc:<10}  {pct:5.1f}%")

        prec_df = pd.DataFrame(precursor_results).fillna(0)
        if not prec_df.empty:
            fig, axes = plt.subplots(1, 2, figsize=(18, 7))
            fig.suptitle(
                f"{DEVICE_TYPE} - Failure Precursor / Leading Indicator"
                f" (look-back={PRECURSOR_WINDOW_DAYS}d)",
                fontsize=13, fontweight="bold"
            )

            sns.heatmap(prec_df, ax=axes[0], cmap="YlOrRd", annot=True, fmt=".0f",
                        linewidths=0.3, cbar_kws={"label": "% of critical events preceded"},
                        annot_kws={"size": 9})
            axes[0].set(
                title="Precursor Heatmap\nrows=precursor code, cols=critical code",
                xlabel="Critical Fault Code", ylabel="Precursor Fault Code"
            )
            axes[0].tick_params(axis="x", rotation=45)

            top_prec = {str(k): v.idxmax() for k, v in precursor_results.items() if len(v) > 0}
            top_pct  = {str(k): v.max()    for k, v in precursor_results.items() if len(v) > 0}
            axes[1].barh(range(len(top_prec)), list(top_pct.values()), color="#1565C0", alpha=0.85)
            axes[1].set_yticks(range(len(top_prec)))
            axes[1].set_yticklabels(
                [f"FC={c}  <-  FC={p}" for c, p in top_prec.items()], fontsize=9
            )
            axes[1].set(title="Strongest Precursor per Critical Fault Code",
                        xlabel="Support (% of critical events)")

            plt.tight_layout()
            plt.show()
    else:
        print("[INFO] No significant precursors found in the data.")
else:
    print("[SKIP] Precursor analysis - insufficient data or no fault code column.")


In [ ]:
# ── Cell 6e — MTBF Aging Trend (Degradation Detection) ───────────────────────
# Rolling quarterly MTBF per device. A declining slope = device is degrading.
# Negative slope -> MTBF shrinking over time -> nearing end of life.
from scipy.stats import linregress as _lr
from matplotlib.patches import Patch as _P2

TOP_N_AGING = 8

if len(df_tbf) > 0 and DATE_COL in df_tbf.columns:
    df_age = df_tbf[[DEV_KEY, DATE_COL, "TBF_days"]].copy()
    df_age[DATE_COL] = pd.to_datetime(df_age[DATE_COL])
    # pandas >= 2.2 uses QE; older uses Q
    for _qalias in ["QE", "Q"]:
        try:
            df_age["period"] = df_age[DATE_COL].dt.to_period(_qalias)
            break
        except Exception:
            continue

    period_mtbf = (
        df_age.groupby([DEV_KEY, "period"])["TBF_days"]
        .mean().reset_index()
    )
    period_mtbf["period_dt"] = period_mtbf["period"].dt.to_timestamp()

    dev_periods  = period_mtbf.groupby(DEV_KEY)["period"].count()
    top_devs_age = dev_periods[dev_periods >= 3].nlargest(TOP_N_AGING).index.tolist()

    slopes = {}
    for dev in dev_periods[dev_periods >= 3].index:
        sub = period_mtbf[period_mtbf[DEV_KEY] == dev].sort_values("period_dt")
        x = (sub["period_dt"] - sub["period_dt"].min()).dt.days.values.astype(float)
        y = sub["TBF_days"].values.astype(float)
        if len(x) >= 2 and x.std() > 0:
            slope, _, _, _, _ = _lr(x, y)
            slopes[dev] = slope

    slope_df = (
        pd.DataFrame.from_dict(slopes, orient="index", columns=["slope_days_per_day"])
        .reset_index().rename(columns={"index": DEV_KEY})
        .sort_values("slope_days_per_day")
    )

    print("=" * 60)
    print(f"MTBF AGING TREND - {DEVICE_TYPE}  (quarterly periods)")
    print("=" * 60)
    n_deg = (slope_df["slope_days_per_day"] < 0).sum()
    n_imp = (slope_df["slope_days_per_day"] > 0).sum()
    print(f"Devices analysed     : {len(slope_df)}")
    print(f"Degrading (slope < 0): {n_deg} ({n_deg/max(len(slope_df),1)*100:.1f}%)")
    print(f"Improving (slope > 0): {n_imp}")
    print("\nFastest Degrading Devices:")
    print(slope_df.head(10).round(4).to_string(index=False))

    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    fig.suptitle(
        f"{DEVICE_TYPE} - MTBF Aging Trend / Degradation Detection",
        fontsize=13, fontweight="bold"
    )

    pal = sns.color_palette("tab10", TOP_N_AGING)
    for i, dev in enumerate(top_devs_age):
        sub = period_mtbf[period_mtbf[DEV_KEY] == dev].sort_values("period_dt")
        axes[0].plot(sub["period_dt"], sub["TBF_days"], "o-", ms=5, lw=1.5,
                     label=f"Dev {dev}", color=pal[i])
        if dev in slopes and len(sub) >= 3:
            x_d   = (sub["period_dt"] - sub["period_dt"].min()).dt.days.values.astype(float)
            y_hat = slopes[dev] * x_d + sub["TBF_days"].iloc[0]
            axes[0].plot(sub["period_dt"], y_hat, "--", lw=1, color=pal[i], alpha=0.5)

    axes[0].set(title="Rolling MTBF per Device (solid=actual, dashed=trend)",
                xlabel="Quarter", ylabel="MTBF (days)")
    axes[0].legend(fontsize=7, ncol=2)
    axes[0].tick_params(axis="x", rotation=30)

    _top_slope = pd.concat([slope_df.head(10), slope_df.tail(10)]).drop_duplicates()
    _c2 = ["#E53935" if s < 0 else "#2E7D32" for s in _top_slope["slope_days_per_day"]]
    axes[1].barh(range(len(_top_slope)), _top_slope["slope_days_per_day"].values,
                 color=_c2, alpha=0.85)
    axes[1].set_yticks(range(len(_top_slope)))
    axes[1].set_yticklabels([str(d) for d in _top_slope[DEV_KEY]], fontsize=8)
    axes[1].axvline(0, color="black", lw=1)
    axes[1].set(title="MTBF Trend Slope per Device\nRed=Degrading  Green=Improving",
                xlabel="MTBF slope (days/day)")
    axes[1].legend(
        handles=[_P2(color="#E53935", label="Degrading (MTBF falling)"),
                 _P2(color="#2E7D32", label="Improving (MTBF rising)")],
        fontsize=8
    )

    plt.tight_layout()
    plt.show()

    DEGRADING_DEVICES = slope_df[slope_df["slope_days_per_day"] < 0][DEV_KEY].tolist()
    print(f"\nDegrading devices: {len(DEGRADING_DEVICES)} stored in DEGRADING_DEVICES.")
else:
    print("[SKIP] MTBF aging trend - insufficient TBF data.")
    DEGRADING_DEVICES = []


In [ ]:
# ── Cell 7 ───────────────────────────────────────────────────────────
# ── Kaplan-Meier Fitter (Overall) ─────────────────────────────────────────────
if len(df_tbf) == 0:
    print("[SKIP] No TBF data.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(f"{DEVICE_TYPE} — Kaplan-Meier Fitter (Overall)", fontsize=14, fontweight="bold")

    # Overall KM
    kmf_all = KaplanMeierFitter()
    kmf_all.fit(df_tbf["TBF_days"], event_observed=df_tbf["event"], label=f"All {DEVICE_TYPE}")
    kmf_all.plot_survival_function(ax=axes[0], ci_show=True, color="#1565C0")
    median_all = kmf_all.median_survival_time_
    axes[0].axhline(0.5, color="red", ls="--", lw=1, label=f"50% at {median_all:.1f}d")
    axes[0].set(title=f"KM Survival — All {DEVICE_TYPE} (median TBF={median_all:.1f}d)",
                xlabel="Days Between Failures", ylabel="Survival Probability")
    axes[0].legend()

    # TBF distribution
    axes[1].hist(df_tbf["TBF_days"].clip(upper=df_tbf["TBF_days"].quantile(0.99)),
                 bins=60, color="#1565C0", alpha=0.75, edgecolor="white")
    axes[1].set(title="Time Between Failures Distribution", xlabel="TBF (days)", ylabel="Count")
    axes[1].axvline(median_all, color="red", ls="--", lw=1.5, label=f"Median={median_all:.1f}d")
    axes[1].legend()

    plt.tight_layout()
    plt.show()
    print(f"Overall KM median TBF: {median_all:.2f} days")


In [ ]:
# ── Cell 8 ───────────────────────────────────────────────────────────
# ── Kaplan-Meier Survival Analysis by Fault Code ──────────────────────────────
# Consider only top 10 fault codes by frequency
if len(df_tbf) == 0 or not TOP10_FC:
    print("[SKIP] No TBF data or fault codes.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    fig.suptitle(f"{DEVICE_TYPE} — Kaplan-Meier by Fault Code (Top {TOP_N})",
                 fontsize=14, fontweight="bold")

    pal = sns.color_palette("tab10", len(TOP10_FC))
    km_medians = {}
    for i, fc in enumerate(TOP10_FC):
        sub = df_tbf[df_tbf[FC_COL] == fc]
        if len(sub) < 5: continue
        kmf = KaplanMeierFitter()
        kmf.fit(sub["TBF_days"], sub["event"], label=f"FC={fc}")
        kmf.plot_survival_function(ax=axes[0], ci_show=False, color=pal[i], lw=1.5)
        km_medians[fc] = kmf.median_survival_time_

    axes[0].axhline(0.5, color="black", ls=":", lw=1)
    axes[0].set(title="KM Survival by Fault Code", xlabel="Days", ylabel="Survival Probability")
    axes[0].legend(fontsize=7, ncol=2)

    # Bar chart of median TBF per fault code
    med_df = pd.Series(km_medians).sort_values()
    colors = [pal[TOP10_FC.index(fc)] if fc in TOP10_FC else "gray" for fc in med_df.index]
    axes[1].barh(range(len(med_df)), med_df.values, color=colors, alpha=0.85)
    axes[1].set_yticks(range(len(med_df)))
    axes[1].set_yticklabels([f"FC={fc}" for fc in med_df.index], fontsize=9)
    axes[1].set(title="Median TBF per Fault Code (days)", xlabel="Median TBF (days)")

    plt.tight_layout()
    plt.show()

    # Log-rank test between top-2 fault codes
    if len(TOP10_FC) >= 2:
        fc1, fc2 = TOP10_FC[0], TOP10_FC[1]
        s1 = df_tbf[df_tbf[FC_COL] == fc1]
        s2 = df_tbf[df_tbf[FC_COL] == fc2]
        if len(s1) >= 5 and len(s2) >= 5:
            lr = logrank_test(s1["TBF_days"], s2["TBF_days"],
                              event_observed_A=s1["event"], event_observed_B=s2["event"])
            print(f"\nLog-rank test FC={fc1} vs FC={fc2}:")
            print(f"  p-value = {lr.p_value:.6f}  (significant if < 0.05)")


In [ ]:
# ── Cell 9 ───────────────────────────────────────────────────────────
# ── Kaplan-Meier Analysis for a Specific Device ───────────────────────────────
if len(df_tbf) == 0 or SELECTED_DEVICE is None:
    print("[SKIP]")
else:
    sub_dev = df_tbf[df_tbf[DEV_KEY] == SELECTED_DEVICE]
    print(f"Device {SELECTED_DEVICE}: {len(sub_dev):,} TBF intervals")

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(f"{DEVICE_TYPE} — KM for Device {SELECTED_DEVICE} (highest-failure device)",
                 fontsize=13, fontweight="bold")

    # Overall KM for selected device
    kmf_dev = KaplanMeierFitter()
    kmf_dev.fit(sub_dev["TBF_days"], sub_dev["event"], label=f"Device {SELECTED_DEVICE}")
    kmf_dev.plot_survival_function(ax=axes[0], ci_show=True, color="#E53935")
    med_dev = kmf_dev.median_survival_time_
    axes[0].axhline(0.5, color="black", ls="--", lw=1, label=f"50% at {med_dev:.1f}d")
    axes[0].set(title=f"KM Survival — Device {SELECTED_DEVICE}  median={med_dev:.1f}d",
                xlabel="Days", ylabel="Survival Probability")
    axes[0].legend()

    # KM per fault code for selected device
    fc_in_dev = sub_dev[FC_COL].value_counts().head(5).index.tolist() if FC_COL else []
    pal = sns.color_palette("tab10", 5)
    for i, fc in enumerate(fc_in_dev):
        s = sub_dev[sub_dev[FC_COL] == fc]
        if len(s) >= 3:
            KaplanMeierFitter().fit(s["TBF_days"], s["event"],
                                    label=f"FC={fc}").plot_survival_function(
                ax=axes[1], ci_show=False, color=pal[i])
    axes[1].set(title=f"KM by Fault Code — Device {SELECTED_DEVICE}",
                xlabel="Days", ylabel="Survival Prob")
    axes[1].legend(fontsize=8)

    plt.tight_layout()
    plt.show()


In [ ]:
# ── Cell 10 ──────────────────────────────────────────────────────────
# ── Weibull Fitting for Different Fault Codes ─────────────────────────────────
# Display Weibull parameters | β (Shape Parameter) Analysis
# Consider only top 10 fault codes by frequency
if len(df_tbf) == 0 or not TOP10_FC:
    print("[SKIP]")
else:
    wb_params = []
    for fc in TOP10_FC:
        sub = df_tbf[df_tbf[FC_COL] == fc]
        if len(sub) < 5: continue
        try:
            wf = WeibullFitter()
            wf.fit(sub["TBF_days"], sub["event"])
            beta  = wf.rho_
            eta   = wf.lambda_ ** (-1/beta) if beta > 0 else float("nan")
            med   = wf.median_survival_time_
            ftype = ("Infant mortality (β<1)" if beta < 1
                     else "Random failures (β≈1)" if abs(beta-1) < 0.15
                     else "Wear-out failures (β>1)")
            wb_params.append({"FaultCode": fc, "Count": len(sub),
                              "β (shape)": round(beta, 4), "η (scale,d)": round(eta, 2),
                              "Median_TBF_d": round(med, 2), "FailureType": ftype})
        except Exception as e:
            wb_params.append({"FaultCode": fc, "Count": len(sub),
                              "β (shape)": None, "η (scale,d)": None,
                              "Median_TBF_d": None, "FailureType": str(e)[:40]})

    wb_df = pd.DataFrame(wb_params)
    print(f"{'='*70}")
    print(f"WEIBULL PARAMETERS — {DEVICE_TYPE} Top 10 Fault Codes")
    print(f"{'='*70}")
    print(wb_df.to_string(index=False))

    # β Shape Parameter bar chart
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(f"{DEVICE_TYPE} — Weibull Fitting by Fault Code (β Analysis)",
                 fontsize=13, fontweight="bold")

    wb_plot = wb_df.dropna(subset=["β (shape)"])
    colors = ["#E53935" if b < 1 else "#FF6F00" if abs(b-1) < 0.15 else "#1565C0"
              for b in wb_plot["β (shape)"]]
    axes[0].barh(range(len(wb_plot)), wb_plot["β (shape)"].values, color=colors, alpha=0.85)
    axes[0].axvline(1.0, color="black", ls="--", lw=1.5, label="β=1 (Random)")
    axes[0].set_yticks(range(len(wb_plot)))
    axes[0].set_yticklabels([f"FC={fc}" for fc in wb_plot["FaultCode"]], fontsize=9)
    axes[0].set(title="β (Shape Parameter) per Fault Code", xlabel="β")
    axes[0].legend()
    from matplotlib.patches import Patch as _P
    axes[0].legend(handles=[_P(color="#E53935",label="β<1 Infant"),
                             _P(color="#FF6F00",label="β≈1 Random"),
                             _P(color="#1565C0",label="β>1 Wear-out")])

    # KM survival curves for top 10 fault codes
    pal = sns.color_palette("tab10", len(TOP10_FC))
    for i, fc in enumerate(TOP10_FC):
        sub = df_tbf[df_tbf[FC_COL] == fc]
        if len(sub) >= 5:
            wf2 = WeibullFitter()
            wf2.fit(sub["TBF_days"], sub["event"], label=f"FC={fc}")
            wf2.plot_survival_function(ax=axes[1], ci_show=False, color=pal[i], lw=1.5)
    axes[1].set(title="Weibull Survival Curves by Fault Code", xlabel="Days", ylabel="Survival Prob")
    axes[1].legend(fontsize=7, ncol=2)

    plt.tight_layout()
    plt.show()


In [ ]:
# ── Cell 10b — Optimal PM Interval from Weibull Parameters ───────────────────
# For each fault code Weibull fit, compute the PM interval T* that achieves
# a target reliability R:  T* = eta x (-ln(R))^(1/beta)
# beta > 1 (wear-out): schedule PM at T* to prevent failure.
# beta <= 1 (random/infant): PM has little value — focus on MTTR instead.
R_TARGETS = [0.99, 0.95, 0.90]

_wb_source = None
if "wb_df" in dir() and len(wb_df) > 0:
    _wb_source = wb_df.copy()
elif "wb_params" in dir() and len(wb_params) > 0:
    _wb_source = pd.DataFrame(wb_params)

if _wb_source is not None:
    pm_rows = []
    _beta_col = next((c for c in _wb_source.columns
                      if "beta" in c.lower() or "shape" in c.lower()), None)
    _eta_col  = next((c for c in _wb_source.columns
                      if "eta" in c.lower() or "scale" in c.lower()), None)
    _fc_col   = next((c for c in _wb_source.columns
                      if "fault" in c.lower() or "code" in c.lower() or c == "FaultCode"), None)

    if _beta_col and _eta_col and _fc_col:
        for _, row in _wb_source.iterrows():
            try:
                beta = float(row[_beta_col]); eta = float(row[_eta_col])
                fc   = row[_fc_col]
            except (TypeError, ValueError):
                continue
            if beta <= 0 or eta <= 0:
                continue
            r = {
                "FaultCode"  : fc,
                "Count"      : int(row.get("Count", 0)),
                "beta"       : round(beta, 3),
                "eta_days"   : round(eta, 2),
                "FailureMode": ("Infant" if beta < 0.85 else "Random" if beta < 1.15 else "Wear-out"),
            }
            for rt in R_TARGETS:
                t_star = eta * ((-np.log(rt)) ** (1.0 / beta))
                r[f"PM@{int(rt*100)}pct_days"] = round(t_star, 1)
            pm_rows.append(r)
    else:
        print(f"[WARN] Could not identify beta/eta/FC columns. Available: {list(_wb_source.columns)}")

    if pm_rows:
        pm_df = pd.DataFrame(pm_rows)
        print("=" * 75)
        print(f"OPTIMAL PM INTERVAL - {DEVICE_TYPE}  [T* = eta x (-ln(R))^(1/beta)]")
        print("=" * 75)
        print(pm_df.to_string(index=False))

        fig, axes = plt.subplots(1, 2, figsize=(18, 7))
        fig.suptitle(f"{DEVICE_TYPE} - Optimal PM Interval per Fault Code",
                     fontsize=13, fontweight="bold")

        x = np.arange(len(pm_df)); w = 0.25
        axes[0].bar(x - w, pm_df["PM@99pct_days"], w, label="99% reliability",
                    color="#1565C0", alpha=0.85)
        axes[0].bar(x,     pm_df["PM@95pct_days"], w, label="95% reliability",
                    color="#FF6F00", alpha=0.85)
        axes[0].bar(x + w, pm_df["PM@90pct_days"], w, label="90% reliability",
                    color="#E53935", alpha=0.85)
        axes[0].set_xticks(x)
        axes[0].set_xticklabels([f"FC={fc}" for fc in pm_df["FaultCode"]],
                                 rotation=45, fontsize=8)
        axes[0].set(title="PM Interval by Reliability Target", ylabel="PM Interval (days)")
        axes[0].legend()

        _sc = axes[1].scatter(pm_df["beta"], pm_df["PM@95pct_days"],
                              c=pm_df["Count"], cmap="Blues", s=80,
                              edgecolors="gray", alpha=0.85)
        plt.colorbar(_sc, ax=axes[1], label="Failure Count")
        axes[1].axvline(1.0, color="black", ls="--", lw=1, label="beta=1 (Random)")
        for _, r in pm_df.iterrows():
            axes[1].annotate(str(r["FaultCode"]), (r["beta"], r["PM@95pct_days"]),
                             fontsize=7, xytext=(4, 2), textcoords="offset points")
        axes[1].set(
            title="PM Interval @95% vs beta Shape\nbeta>1 Wear-out: shorter PM; beta<1 Infant: PM less effective",
            xlabel="beta (Weibull shape)", ylabel="PM Interval @95% (days)"
        )
        axes[1].legend(fontsize=8)

        plt.tight_layout()
        plt.show()

        print("\nInterpretation:")
        print("  beta > 1 (wear-out) : PM interval = T* — schedule BEFORE failure.")
        print("  beta ~ 1 (random)   : PM has minimal impact — improve MTTR instead.")
        print("  beta < 1 (infant)   : Replace at installation; PM does not help.")
    else:
        print("[SKIP] No valid Weibull parameters - check wb_df column names.")
else:
    print("[SKIP] Weibull parameters not found - run Weibull Fitting cell first.")


In [ ]:
# ── Cell 11 ──────────────────────────────────────────────────────────
# ── Weibull Fitting for the Selected Device ────────────────────────────────────
if len(df_tbf) == 0 or SELECTED_DEVICE is None:
    print("[SKIP]")
else:
    sub_dev = df_tbf[df_tbf[DEV_KEY] == SELECTED_DEVICE]
    if len(sub_dev) < 5:
        print(f"[SKIP] Only {len(sub_dev)} TBF records for device {SELECTED_DEVICE}.")
    else:
        wf = WeibullFitter()
        wf.fit(sub_dev["TBF_days"], sub_dev["event"])
        beta  = wf.rho_
        eta   = wf.lambda_ ** (-1/beta) if beta > 0 else float("nan")
        med   = wf.median_survival_time_
        ftype = ("Infant mortality (β<1)" if beta < 1
                 else "Random failures (β≈1)" if abs(beta-1) < 0.15
                 else "Wear-out failures (β>1)")

        print(f"Weibull for Device {SELECTED_DEVICE}:")
        print(f"  β (shape) = {beta:.4f}  ->  {ftype}")
        print(f"  η (scale) = {eta:.2f} days")
        print(f"  Median TBF = {med:.2f} days")

        fig, axes = plt.subplots(1, 3, figsize=(20, 6))
        fig.suptitle(f"{DEVICE_TYPE} — Weibull for Device {SELECTED_DEVICE}  β={beta:.3f}  [{ftype.split('(')[0].strip()}]",
                     fontsize=13, fontweight="bold")

        wf.plot_survival_function(ax=axes[0], color="#E53935")
        axes[0].set(title=f"Weibull Survival  β={beta:.3f}", xlabel="Days")

        wf.plot_cumulative_hazard(ax=axes[1], color="#1565C0")
        axes[1].set(title="Cumulative Hazard", xlabel="Days")

        tbf_clip = sub_dev["TBF_days"].clip(upper=sub_dev["TBF_days"].quantile(0.99))
        axes[2].hist(tbf_clip, bins=40, density=True, color="#FFA726", alpha=0.75, edgecolor="white", label="TBF")
        if beta > 0 and not np.isnan(eta):
            x = np.linspace(0.1, tbf_clip.max(), 300)
            axes[2].plot(x, weibull_min.pdf(x, beta, scale=eta), "b-", lw=2, label="Weibull fit")
        axes[2].set(title="TBF Distribution + Weibull Fit", xlabel="Days", ylabel="Density")
        axes[2].legend()

        plt.tight_layout()
        plt.show()


In [ ]:
# ── Cell 12 ──────────────────────────────────────────────────────────
# ── Weibull Failure Prediction per Fault Code ──────────────────────────────────
# Statistical prediction of expected time to next failure using Weibull distribution
if len(df_tbf) == 0 or not TOP10_FC:
    print("[SKIP]")
else:
    pred_rows = []
    for fc in TOP10_FC:
        sub = df_tbf[df_tbf[FC_COL] == fc]
        if len(sub) < 5: continue
        try:
            wf = WeibullFitter()
            wf.fit(sub["TBF_days"], sub["event"])
            beta = wf.rho_
            eta  = wf.lambda_ ** (-1/beta) if beta > 0 else float("nan")
            # Weibull predictions: prob of surviving t=7, 14, 30, 60 days
            surv_7d  = float(wf.survival_function_at_times([7]).iloc[0])   if not np.isnan(eta) else float("nan")
            surv_30d = float(wf.survival_function_at_times([30]).iloc[0])  if not np.isnan(eta) else float("nan")
            med      = wf.median_survival_time_
            pred_rows.append({"FaultCode": fc, "Count": len(sub),
                              "β": round(beta, 3), "Median_TBF_d": round(med, 1),
                              "P(survive>7d)": round(surv_7d, 4),
                              "P(survive>30d)": round(surv_30d, 4),
                              "P(fail<=7d)": round(1 - surv_7d, 4),
                              "P(fail<=30d)": round(1 - surv_30d, 4)})
        except Exception: pass

    if pred_rows:
        pred_df = pd.DataFrame(pred_rows)
        print(f"{'='*75}")
        print(f"WEIBULL FAILURE PREDICTION — {DEVICE_TYPE} (probability of failure within horizon)")
        print(f"{'='*75}")
        print(pred_df.to_string(index=False))

        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        fig.suptitle(f"{DEVICE_TYPE} — Weibull Failure Prediction per Fault Code",
                     fontsize=13, fontweight="bold")

        pal = sns.color_palette("tab10", len(pred_df))
        axes[0].barh(range(len(pred_df)), pred_df["P(fail<=7d)"], color="#E53935", alpha=0.85)
        axes[0].set_yticks(range(len(pred_df)))
        axes[0].set_yticklabels([f"FC={r}" for r in pred_df["FaultCode"]], fontsize=9)
        axes[0].set(title="P(Failure within 7 days) per Fault Code", xlabel="Probability")

        axes[1].barh(range(len(pred_df)), pred_df["P(fail<=30d)"], color="#FF6F00", alpha=0.85)
        axes[1].set_yticks(range(len(pred_df)))
        axes[1].set_yticklabels([f"FC={r}" for r in pred_df["FaultCode"]], fontsize=9)
        axes[1].set(title="P(Failure within 30 days) per Fault Code", xlabel="Probability")

        plt.tight_layout()
        plt.show()


In [ ]:
# ── Cell 13 ──────────────────────────────────────────────────────────
# ── Cox Proportional Hazard Model ─────────────────────────────────────────────
# Fault codes as one-hot covariates; additional: is_chargeable, priority, time_to_resolve
if len(df_tbf) == 0 or not TOP10_FC:
    print("[SKIP]")
else:
    # One-hot encode top-10 fault codes
    df_cox = df_tbf[[DEV_KEY, FC_COL, "TBF_days", "event"]].copy()
    for fc in TOP10_FC:
        df_cox[f"FC_{fc}"] = (df_cox[FC_COL] == fc).astype(int)

    # Add numeric covariates if available
    cov_candidates = ["is_chargeable", "priority", "time_to_resolve_minutes", "is_major_incident"]
    extra_cov = [c for c in cov_candidates if c in df_tbf.columns]
    if extra_cov:
        df_cox = df_cox.merge(df_tbf[extra_cov + [df_tbf.index.name or "idx"]].reset_index().rename(columns={"index":"_idx"}),
                               left_index=True, right_on="_idx", how="left")
        df_cox = df_cox.merge(df_tbf[extra_cov].reset_index(drop=True),
                               left_index=True, right_index=True, how="left")
    # Cleaner: rebuild from df_sorted
    for col in cov_candidates:
        if col in df_sorted.columns:
            df_cox[col] = df_sorted.loc[df_tbf.index, col].values if len(df_sorted.loc[df_tbf.index]) == len(df_cox) else df_tbf.get(col, np.nan)

    fc_cols = [f"FC_{fc}" for fc in TOP10_FC if f"FC_{fc}" in df_cox.columns]
    num_cov = [c for c in cov_candidates if c in df_cox.columns]
    all_cov = fc_cols + num_cov
    df_cox[all_cov] = df_cox[all_cov].fillna(0)

    # Scale numeric covariates
    if num_cov:
        sc = StandardScaler()
        df_cox[num_cov] = sc.fit_transform(df_cox[num_cov])

    cox_data = df_cox[["TBF_days", "event"] + all_cov].dropna()
    cph = CoxPHFitter(penalizer=0.1)
    cph.fit(cox_data, duration_col="TBF_days", event_col="event")

    print("Cox Proportional Hazard Model Summary:")
    print(cph.summary[["coef", "exp(coef)", "p", "coef lower 95%", "coef upper 95%"]].round(4).to_string())
    print(f"\nConcordance index: {cph.concordance_index_:.4f}")
    print(f"Log-likelihood    : {cph.log_likelihood_:.4f}")

    # ── Plot Cox PH Model results ──────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    fig.suptitle(f"{DEVICE_TYPE} — Cox PH Model  CI={cph.concordance_index_:.3f}",
                 fontsize=13, fontweight="bold")

    coef = cph.summary["coef"].sort_values()
    colors = ["#E53935" if v > 0 else "#1565C0" for v in coef.values]
    axes[0].barh(range(len(coef)), coef.values, color=colors, alpha=0.85)
    axes[0].set_yticks(range(len(coef)))
    axes[0].set_yticklabels([c[:20] for c in coef.index], fontsize=8)
    axes[0].axvline(0, color="black", lw=1)
    axes[0].set(title="Cox Coefficients (log Hazard Ratios)\nRed=increases risk, Blue=decreases risk",
                xlabel="Coefficient")

    # Partial effect of most influential feature
    top_feat = coef.abs().idxmax()
    try:
        cph.plot_partial_effects_on_outcome(top_feat, values=[-1, 0, 1], ax=axes[1])
        axes[1].set(title=f"Partial Effect of '{top_feat}'")
    except Exception as e:
        axes[1].text(0.5, 0.5, f"Plot error:\n{e}", ha="center", va="center",
                    transform=axes[1].transAxes, fontsize=9)
        axes[1].set_title(f"Partial Effect — {top_feat}")

    plt.tight_layout()
    plt.show()


In [ ]:
# ── Cell 14 ──────────────────────────────────────────────────────────
# ── Failure Rate Over Time per Fault Code ─────────────────────────────────────
if len(df_tbf) == 0 or not TOP10_FC:
    print("[SKIP]")
else:
    fig, ax = plt.subplots(figsize=(16, 7))
    fig.suptitle(f"{DEVICE_TYPE} — Failure Rate Over Time per Fault Code (Top {min(5, len(TOP10_FC))})",
                 fontsize=13, fontweight="bold")

    pal = sns.color_palette("tab10", len(TOP10_FC))
    for i, fc in enumerate(TOP10_FC[:5]):
        sub = df[df[FC_COL] == fc].copy()
        if len(sub) == 0: continue
        monthly = sub.groupby(sub[DATE_COL].dt.to_period("M")).size()
        ax.plot(range(len(monthly)), monthly.values, marker="o", ms=4, lw=1.5,
                label=f"FC={fc}", color=pal[i])

    # x-axis labels
    _sample_fc = TOP10_FC[0]
    _sample_sub = df[df[FC_COL] == _sample_fc]
    if len(_sample_sub) > 0:
        _months = _sample_sub.groupby(_sample_sub[DATE_COL].dt.to_period("M")).size().index
        step = max(1, len(_months) // 8)
        ax.set_xticks(range(0, len(_months), step))
        ax.set_xticklabels([str(m) for m in _months[::step]], rotation=45, fontsize=8)

    ax.set(title="Monthly Failure Count per Fault Code", xlabel="Month", ylabel="Failure Count")
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()


In [ ]:
# ── Cell 15 ──────────────────────────────────────────────────────────
# ── Hazard Function Estimation per Fault Code (Nelson-Aalen) ──────────────────
if len(df_tbf) == 0 or not TOP10_FC:
    print("[SKIP]")
else:
    fig, ax = plt.subplots(figsize=(16, 7))
    fig.suptitle(f"{DEVICE_TYPE} — Nelson-Aalen Hazard Function per Fault Code (Top {TOP_N})",
                 fontsize=13, fontweight="bold")

    pal = sns.color_palette("tab10", len(TOP10_FC))
    for i, fc in enumerate(TOP10_FC):
        sub = df_tbf[df_tbf[FC_COL] == fc]
        if len(sub) < 5: continue
        naf = NelsonAalenFitter()
        naf.fit(sub["TBF_days"], sub["event"], label=f"FC={fc}")
        naf.plot_cumulative_hazard(ax=ax, ci_show=False, color=pal[i], lw=1.5)

    ax.set(title="Cumulative Hazard Function per Fault Code", xlabel="Days Between Failures")
    ax.legend(fontsize=8, ncol=2)
    plt.tight_layout()
    plt.show()


In [ ]:
# ── Cell 16 ──────────────────────────────────────────────────────────
# ── Method 1: Multi-Factor Risk Ranking ───────────────────────────────────────
if len(df_tbf) == 0:
    print("[SKIP]")
else:
    dev_stats = df_tbf.groupby(DEV_KEY).agg(
        failure_count=("TBF_days", "count"),
        avg_tbf=("TBF_days", "mean"),
        min_tbf=("TBF_days", "min"),
    ).reset_index()
    dev_stats["MTBF_days"] = dev_stats["avg_tbf"]

    def _norm(s): rng = s.max()-s.min(); return (s-s.min())/rng if rng>0 else s*0
    dev_stats["risk_score"] = (
        _norm(dev_stats["failure_count"]) +
        (1 - _norm(dev_stats["MTBF_days"])) +
        (1 - _norm(dev_stats["min_tbf"]))
    ) / 3.0

    top10_risk = dev_stats.nlargest(10, "risk_score")
    print(f"{'='*60}")
    print(f"Method 1: Multi-Factor Risk Ranking — Top 10 {DEVICE_TYPE} Devices")
    print(f"{'='*60}")
    print(top10_risk[[DEV_KEY, "failure_count", "MTBF_days", "min_tbf", "risk_score"]].round(3).to_string(index=False))

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(f"{DEVICE_TYPE} — Method 1: Multi-Factor Risk Ranking", fontsize=13, fontweight="bold")

    axes[0].barh(range(len(top10_risk)), top10_risk["risk_score"].values, color="#E53935", alpha=0.85)
    axes[0].set_yticks(range(len(top10_risk)))
    axes[0].set_yticklabels([str(d) for d in top10_risk[DEV_KEY]], fontsize=9)
    axes[0].set(title="Top 10 High-Risk Devices (Multi-Factor Score)", xlabel="Risk Score")

    axes[1].scatter(dev_stats["MTBF_days"], dev_stats["failure_count"],
                    c=dev_stats["risk_score"], cmap="RdYlGn_r", alpha=0.6, s=30, edgecolors="none")
    axes[1].set(title="Failure Count vs MTBF (Risk color)", xlabel="MTBF (days)", ylabel="Failure Count")
    plt.colorbar(axes[1].collections[0], ax=axes[1], label="Risk Score")

    plt.tight_layout()
    plt.show()


In [ ]:
# ── Cell 17 ──────────────────────────────────────────────────────────
# ── MTBF-Based Risk Ranking ───────────────────────────────────────────────────
if len(df_tbf) == 0:
    print("[SKIP]")
else:
    # Risk score = 1/MTBF (normalized)
    dev_stats["mtbf_risk"] = _norm(1.0 / dev_stats["MTBF_days"].replace(0, np.nan).fillna(dev_stats["MTBF_days"].max()))
    top10_mtbf = dev_stats.nlargest(10, "mtbf_risk")

    print(f"{'='*60}")
    print(f"MTBF-Based Risk Ranking — Top 10 {DEVICE_TYPE} Devices")
    print(f"{'='*60}")
    print(top10_mtbf[[DEV_KEY, "failure_count", "MTBF_days", "mtbf_risk"]].round(3).to_string(index=False))

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(f"{DEVICE_TYPE} — MTBF-Based Risk Ranking", fontsize=13, fontweight="bold")

    axes[0].hist(dev_stats["MTBF_days"].clip(upper=dev_stats["MTBF_days"].quantile(0.99)),
                 bins=40, color="#1565C0", alpha=0.8, edgecolor="white")
    med_mtbf = dev_stats["MTBF_days"].median()
    axes[0].axvline(med_mtbf, color="red", ls="--", lw=1.5, label=f"Median={med_mtbf:.1f}d")
    axes[0].set(title="MTBF Distribution Across All Devices", xlabel="MTBF (days)", ylabel="Devices")
    axes[0].legend()

    axes[1].barh(range(len(top10_mtbf)), top10_mtbf["MTBF_days"].values, color="#FF6F00", alpha=0.85)
    axes[1].set_yticks(range(len(top10_mtbf)))
    axes[1].set_yticklabels([str(d) for d in top10_mtbf[DEV_KEY]], fontsize=9)
    axes[1].axvline(med_mtbf, color="red", ls="--", lw=1, label=f"Fleet median={med_mtbf:.1f}d")
    axes[1].set(title="Shortest MTBF Devices (highest failure frequency)", xlabel="MTBF (days)")
    axes[1].legend(fontsize=8)

    plt.tight_layout()
    plt.show()


In [ ]:
# ── Cell 18 ──────────────────────────────────────────────────────────
# ── Device + Fault Code Risk — Failure Reasons in Depth ──────────────────────
# Which fault codes are most problematic per device?
if len(df_tbf) == 0 or not FC_COL:
    print("[SKIP]")
else:
    # MTBF per (Device, FaultCode) pair
    dev_fc_stats = df_tbf.groupby([DEV_KEY, FC_COL]).agg(
        count=("TBF_days", "count"),
        avg_tbf=("TBF_days", "mean"),
    ).reset_index()
    dev_fc_stats["risk_score"] = _norm(dev_fc_stats["count"]) + (1 - _norm(dev_fc_stats["avg_tbf"]))

    top20_pairs = dev_fc_stats.nlargest(20, "risk_score")
    print(f"{'='*65}")
    print(f"Device + Fault Code Risk — Top 20 Pairs  ({DEVICE_TYPE})")
    print(f"{'='*65}")
    print(top20_pairs[[DEV_KEY, FC_COL, "count", "avg_tbf", "risk_score"]].round(3).to_string(index=False))

    fig, axes = plt.subplots(1, 2, figsize=(18, 8))
    fig.suptitle(f"{DEVICE_TYPE} — Device + Fault Code Risk (Failure Reasons per Device)",
                 fontsize=13, fontweight="bold")

    labels = [f"{r[DEV_KEY]}\nFC={r[FC_COL]}" for _, r in top20_pairs.iterrows()]
    axes[0].barh(range(len(top20_pairs)), top20_pairs["risk_score"].values, color="#E53935", alpha=0.85)
    axes[0].set_yticks(range(len(top20_pairs)))
    axes[0].set_yticklabels(labels, fontsize=7)
    axes[0].set(title="Top 20 Device+FaultCode Risk Pairs", xlabel="Risk Score")

    axes[1].barh(range(len(top20_pairs)), top20_pairs["avg_tbf"].values, color="#FF6F00", alpha=0.85)
    axes[1].set_yticks(range(len(top20_pairs)))
    axes[1].set_yticklabels(labels, fontsize=7)
    axes[1].set(title="Avg TBF per Device+FaultCode Pair (days)", xlabel="Avg TBF (days)")

    plt.tight_layout()
    plt.show()


In [ ]:
# ── Cell 19 ──────────────────────────────────────────────────────────
# ── Severity Levels — Device-Level Risk Ranking ───────────────────────────────
# Severity Score = Nelson-Aalen cumulative hazard + (1 - normalized MTBF)
if len(df_tbf) == 0:
    print("[SKIP]")
else:
    top_devs = df_tbf[DEV_KEY].value_counts().nlargest(TOP_N).index.tolist()
    dev_hazard = {}
    for dev in top_devs:
        sub = df_tbf[df_tbf[DEV_KEY] == dev]
        if len(sub) >= 3:
            naf = NelsonAalenFitter()
            naf.fit(sub["TBF_days"], sub["event"])
            dev_hazard[dev] = float(naf.cumulative_hazard_.iloc[-1, 0])

    if dev_hazard:
        sev = pd.DataFrame.from_dict(dev_hazard, orient="index", columns=["cum_hazard"])
        sev.index.name = DEV_KEY; sev = sev.reset_index()
        sev = sev.merge(dev_stats[[DEV_KEY, "MTBF_days", "failure_count"]], on=DEV_KEY, how="left")
        sev["mtbf_norm"] = _norm(sev["MTBF_days"].fillna(sev["MTBF_days"].median()))
        sev["severity"]  = sev["cum_hazard"] + (1 - sev["mtbf_norm"])
        sev = sev.sort_values("severity", ascending=False)

        # Assign severity labels
        q75 = sev["severity"].quantile(0.75)
        q50 = sev["severity"].quantile(0.5)
        sev["severity_level"] = sev["severity"].apply(
            lambda x: "CRITICAL" if x >= q75 else ("HIGH" if x >= q50 else "MEDIUM"))

        print(f"{'='*60}")
        print(f"Severity Levels — {DEVICE_TYPE} Devices")
        print(f"{'='*60}")
        print(sev[[DEV_KEY, "failure_count", "MTBF_days", "cum_hazard", "severity", "severity_level"]].round(3).to_string(index=False))

        fig, ax = plt.subplots(figsize=(12, 7))
        fig.suptitle(f"{DEVICE_TYPE} — Severity Levels: Device-Level Risk Ranking", fontsize=13, fontweight="bold")

        color_map = {"CRITICAL": "#E53935", "HIGH": "#FF6F00", "MEDIUM": "#1565C0"}
        colors = [color_map.get(lvl, "#999") for lvl in sev["severity_level"]]
        ax.barh(range(len(sev)), sev["severity"].values, color=colors, alpha=0.85)
        ax.set_yticks(range(len(sev)))
        ax.set_yticklabels([str(d) for d in sev[DEV_KEY]], fontsize=8)
        ax.set(title="Device Severity Score (Hazard + 1/MTBF)", xlabel="Severity Score")
        from matplotlib.patches import Patch as _P
        ax.legend(handles=[_P(color="#E53935",label="CRITICAL"),
                           _P(color="#FF6F00",label="HIGH"),
                           _P(color="#1565C0",label="MEDIUM")])
        plt.tight_layout()
        plt.show()


In [ ]:
# ── Cell 20 ──────────────────────────────────────────────────────────
# ── Severity Analysis for Device + Fault Code Risk Ranking ────────────────────
# Nelson-Aalen per (Device, FaultCode) — severity heatmap
if len(df_tbf) == 0 or not FC_COL:
    print("[SKIP]")
else:
    top10_dev_sev = df_tbf[DEV_KEY].value_counts().nlargest(10).index.tolist()
    top5_fc_sev   = df_tbf[FC_COL].value_counts().nlargest(5).index.tolist()
    sev_pairs = []
    for dev in top10_dev_sev:
        for fc in top5_fc_sev:
            sub = df_tbf[(df_tbf[DEV_KEY] == dev) & (df_tbf[FC_COL] == fc)]
            if len(sub) < 3: continue
            naf = NelsonAalenFitter()
            naf.fit(sub["TBF_days"], sub["event"])
            h = float(naf.cumulative_hazard_.iloc[-1, 0])
            mtbf = sub["TBF_days"].mean()
            sev_pairs.append({DEV_KEY: dev, FC_COL: fc, "hazard": h, "mtbf": mtbf,
                              "severity": h + (1 - h / (h + 1))})

    if sev_pairs:
        sev_pair_df = pd.DataFrame(sev_pairs)
        pivot = sev_pair_df.pivot_table(index=DEV_KEY, columns=FC_COL, values="severity", aggfunc="mean")

        fig, ax = plt.subplots(figsize=(14, 7))
        fig.suptitle(f"{DEVICE_TYPE} — Severity Heatmap: Device × Fault Code", fontsize=13, fontweight="bold")
        sns.heatmap(pivot, ax=ax, cmap="YlOrRd", linewidths=0.3, annot=True, fmt=".2f", fontsize=8,
                    cbar_kws={"label": "Severity Score"})
        ax.set(title="Severity by Device × Fault Code", xlabel="Fault Code", ylabel="Device Key")
        ax.tick_params(axis="x", rotation=45)
        plt.tight_layout()
        plt.show()
    else:
        print("[INFO] Not enough data for Device × Fault Code severity heatmap.")


In [ ]:
# ── Cell 21 ──────────────────────────────────────────────────────────
# ── Device Level Fault Occurrences for Top 20 Devices ─────────────────────────
if len(df) == 0 or not FC_COL:
    print("[SKIP]")
else:
    top20_devs = df[DEV_KEY].value_counts().nlargest(20).index.tolist()
    top10_fc_plot = df[FC_COL].value_counts().nlargest(10).index.tolist()

    pivot_occ = df[df[DEV_KEY].isin(top20_devs) & df[FC_COL].isin(top10_fc_plot)]\
        .groupby([DEV_KEY, FC_COL]).size().unstack(fill_value=0)

    fig, ax = plt.subplots(figsize=(16, 9))
    fig.suptitle(f"{DEVICE_TYPE} — Fault Occurrence Heatmap: Top 20 Devices × Top 10 Fault Codes",
                 fontsize=13, fontweight="bold")

    sns.heatmap(pivot_occ, ax=ax, cmap="YlOrRd", linewidths=0.3,
                annot=True, fmt="d", annot_kws={"size": 7},
                cbar_kws={"label": "Occurrence Count"})
    ax.set(title="Fault Occurrences per Device", xlabel="Fault Code", ylabel="Device Key")
    ax.tick_params(axis="x", rotation=45, labelsize=9)
    ax.tick_params(axis="y", labelsize=8)
    plt.tight_layout()
    plt.show()


In [ ]:
# ── Cell 22 ──────────────────────────────────────────────────────────
# ── Crow-AMSAA Reliability Growth Model ───────────────────────────────────────
if len(df) == 0:
    print("[SKIP]")
else:
    fig, axes = plt.subplots(1, 3, figsize=(22, 7))
    fig.suptitle(f"{DEVICE_TYPE} — Crow-AMSAA Reliability Growth + Bathtub Curve + MCF",
                 fontsize=13, fontweight="bold")

    # 1. Crow-AMSAA: cumulative failures per fault code over time
    top5_fc_crow = df[FC_COL].value_counts().nlargest(5).index.tolist() if FC_COL else []
    pal = sns.color_palette("tab10", 5)
    for i, fc in enumerate(top5_fc_crow):
        sub = df[df[FC_COL] == fc].sort_values(DATE_COL)
        monthly = sub.groupby(sub[DATE_COL].dt.to_period("M")).size().cumsum()
        axes[0].plot(range(len(monthly)), monthly.values, "o-", ms=4, lw=1.5,
                     label=f"FC={fc}", color=pal[i])
    axes[0].set(title="Crow-AMSAA: Cumulative Failures per Fault Code",
                xlabel="Month Index", ylabel="Cumulative Failures")
    axes[0].legend(fontsize=8)

    # 2. Bathtub curve: TBF density histogram
    if len(df_tbf) > 0:
        tbf_clip = df_tbf["TBF_days"].clip(upper=df_tbf["TBF_days"].quantile(0.98))
        axes[1].hist(tbf_clip, bins=60, density=True, color="#FF6F00", alpha=0.75, edgecolor="white")
        axes[1].set(title="Bathtub Curve (TBF Density)", xlabel="TBF (days)", ylabel="Density")

    # 3. MCF (Mean Cumulative Function) via Nelson-Aalen for top-5 fault codes
    if len(df_tbf) > 0 and FC_COL:
        for i, fc in enumerate(top5_fc_crow):
            sub = df_tbf[df_tbf[FC_COL] == fc]
            if len(sub) >= 5:
                naf = NelsonAalenFitter()
                naf.fit(sub["TBF_days"], sub["event"], label=f"FC={fc}")
                naf.plot_cumulative_hazard(ax=axes[2], ci_show=False, color=pal[i], lw=1.5)
    axes[2].set(title="MCF (Mean Cumulative Function) per Fault Code",
                xlabel="Days", ylabel="Mean Cumulative Failures")
    axes[2].legend(fontsize=8)

    plt.tight_layout()
    plt.show()


In [ ]:
# ── Cell 23 ──────────────────────────────────────────────────────────
# ── Reliability Block Diagram (RBD) ───────────────────────────────────────────
# NetworkX DiGraph: Device → FaultCode nodes for top-5 devices by failure count
if len(df) == 0 or not FC_COL:
    print("[SKIP]")
else:
    top5_rbd = df[DEV_KEY].value_counts().nlargest(5).index.tolist()

    fig, axes = plt.subplots(1, 2, figsize=(20, 9))
    fig.suptitle(f"{DEVICE_TYPE} — Reliability Block Diagram (Top 5 Devices)",
                 fontsize=13, fontweight="bold")

    # One combined graph for all top-5 devices
    G = nx.DiGraph()
    for dev in top5_rbd:
        sub = df[df[DEV_KEY] == dev]
        fc_counts = sub[FC_COL].value_counts().head(5)
        dev_node = f"DEV\n{dev}"
        G.add_node(dev_node, node_type="device")
        for fc, cnt in fc_counts.items():
            fc_node = f"FC\n{fc}"
            G.add_node(fc_node, node_type="fault")
            G.add_edge(dev_node, fc_node, weight=cnt)

    pos = nx.spring_layout(G, seed=RANDOM_STATE, k=2.5)
    dev_nodes  = [n for n, d in G.nodes(data=True) if d.get("node_type") == "device"]
    fault_nodes = [n for n, d in G.nodes(data=True) if d.get("node_type") == "fault"]
    edge_weights = [G[u][v]["weight"] for u, v in G.edges()]
    max_w = max(edge_weights) if edge_weights else 1

    nx.draw_networkx_nodes(G, pos, nodelist=dev_nodes, node_color="#1565C0",
                           node_size=800, ax=axes[0], alpha=0.9)
    nx.draw_networkx_nodes(G, pos, nodelist=fault_nodes, node_color="#E53935",
                           node_size=500, node_shape="s", ax=axes[0], alpha=0.8)
    nx.draw_networkx_labels(G, pos, ax=axes[0], font_size=5)
    nx.draw_networkx_edges(G, pos, ax=axes[0],
                           width=[2.0 * w/max_w for w in edge_weights],
                           edge_color="#555", arrows=True, arrowsize=15,
                           connectionstyle="arc3,rad=0.1")
    axes[0].set_title("RBD: Device → Fault Code (edge width = occurrence count)", fontsize=10)
    axes[0].axis("off")
    from matplotlib.patches import Patch as _P
    axes[0].legend(handles=[_P(color="#1565C0",label="Device"),
                             _P(color="#E53935",label="Fault Code")], fontsize=9)

    # Fault occurrence + avg TBF table for top-5 devices (heatmap)
    top5_fc_rbd = df[FC_COL].value_counts().nlargest(8).index.tolist()
    sub_rbd = df[df[DEV_KEY].isin(top5_rbd) & df[FC_COL].isin(top5_fc_rbd)]
    pivot_rbd = sub_rbd.groupby([DEV_KEY, FC_COL]).size().unstack(fill_value=0)
    if not pivot_rbd.empty:
        sns.heatmap(pivot_rbd, ax=axes[1], cmap="Blues", annot=True, fmt="d",
                    annot_kws={"size": 9}, linewidths=0.3, cbar_kws={"label": "Count"})
        axes[1].set(title="Fault Occurrences — Top 5 Devices",
                    xlabel="Fault Code", ylabel="Device Key")
        axes[1].tick_params(axis="x", rotation=45)

    plt.tight_layout()
    plt.show()

    # Print fault occurrence + avg TBF per device
    print(f"\nFault Occurrence + Avg TBF per Device (top 5 devices):")
    for dev in top5_rbd:
        sub = df[df[DEV_KEY] == dev]
        fc_occ = sub[FC_COL].value_counts().head(5)
        sub_tbf = df_tbf[df_tbf[DEV_KEY] == dev].groupby(FC_COL)["TBF_days"].mean().round(1)
        print(f"  Device {dev}: {len(sub):,} incidents")
        for fc in fc_occ.index:
            avg_t = sub_tbf.get(fc, float("nan"))
            print(f"    FC={fc}: {fc_occ[fc]:,} occurrences  avg_TBF={avg_t:.1f}d")


In [ ]:
# ── Cell 24 ──────────────────────────────────────────────────────────
# ── RUL Calculation by Different Methods ──────────────────────────────────────
# Method 1: Standard = avg_TBF × (n+1)
# Method 2: Decay Factor = avg_TBF × (n+1) × 0.85
# Method 3: Rolling Window (window=3)
if len(df_tbf) == 0:
    print("[SKIP]")
else:
    rul = df_tbf.sort_values([DEV_KEY, FC_COL, DATE_COL] if DATE_COL in df_tbf.columns else [DEV_KEY, FC_COL]).copy()
    rul["rolling_avg_tbf"] = rul.groupby([DEV_KEY, FC_COL])["TBF_days"].transform(
        lambda x: x.rolling(window=3, min_periods=1).mean())

    # Per device
    rul_dev = df_tbf.groupby(DEV_KEY).agg(
        avg_tbf=("TBF_days", "mean"),
        total_failures=("TBF_days", "count"),
    ).reset_index()
    rul_dev["rolling_tbf"] = df_tbf.sort_values([DEV_KEY, DATE_COL]).groupby(DEV_KEY)["TBF_days"]\
        .transform(lambda x: x.rolling(3, min_periods=1).mean()).groupby(df_tbf[DEV_KEY]).last().reindex(rul_dev[DEV_KEY]).values

    n = rul_dev["total_failures"]
    rul_dev["RUL_standard"] = rul_dev["avg_tbf"] * (n + 1)
    rul_dev["RUL_decay"]    = rul_dev["avg_tbf"] * (n + 1) * 0.85
    rul_dev["RUL_rolling"]  = rul_dev["rolling_tbf"].fillna(rul_dev["avg_tbf"]) * np.minimum(n + 1, 5)

    # Per fault code
    rul_fc = df_tbf.groupby(FC_COL).agg(
        avg_tbf=("TBF_days", "mean"),
        total_failures=("TBF_days", "count"),
    ).reset_index()
    n_fc = rul_fc["total_failures"]
    rul_fc["RUL_standard"] = rul_fc["avg_tbf"] * (n_fc + 1)
    rul_fc["RUL_decay"]    = rul_fc["avg_tbf"] * (n_fc + 1) * 0.85
    rul_fc["RUL_rolling"]  = rul_fc["avg_tbf"] * np.minimum(n_fc + 1, 5)

    top30_dev = rul_dev.nlargest(min(30, len(rul_dev)), "total_failures")
    top10_fc_rul = rul_fc.nlargest(min(10, len(rul_fc)), "total_failures")

    print(f"{'='*70}")
    print(f"RUL Calculation — {DEVICE_TYPE} Devices (top 30 by failure count)")
    print(f"{'='*70}")
    print(top30_dev[[DEV_KEY, "total_failures", "avg_tbf",
                     "RUL_standard", "RUL_decay", "RUL_rolling"]].round(1).to_string(index=False))

    print(f"\n{'='*70}")
    print(f"RUL Calculation — {DEVICE_TYPE} Fault Codes (top 10 by frequency)")
    print(f"{'='*70}")
    print(top10_fc_rul[[FC_COL, "total_failures", "avg_tbf",
                         "RUL_standard", "RUL_decay", "RUL_rolling"]].round(1).to_string(index=False))

    fig, axes = plt.subplots(2, 2, figsize=(20, 14))
    fig.suptitle(f"{DEVICE_TYPE} — RUL Calculation by Different Methods", fontsize=14, fontweight="bold")

    # Device RUL comparison
    x = np.arange(len(top30_dev)); w = 0.28
    axes[0,0].bar(x-w, top30_dev["RUL_standard"], w, label="Standard",   color="#1565C0", alpha=0.85)
    axes[0,0].bar(x,   top30_dev["RUL_decay"],    w, label="Decay ×0.85",color="#FF6F00", alpha=0.85)
    axes[0,0].bar(x+w, top30_dev["RUL_rolling"],  w, label="Rolling",    color="#2E7D32", alpha=0.85)
    axes[0,0].set_xticks(x)
    axes[0,0].set_xticklabels([str(d) for d in top30_dev[DEV_KEY]], rotation=90, fontsize=6)
    axes[0,0].set(title="RUL per Device (top 30 by failure count)", ylabel="RUL (days)")
    axes[0,0].legend()

    # Fault code RUL comparison
    xf = np.arange(len(top10_fc_rul))
    axes[0,1].bar(xf-w, top10_fc_rul["RUL_standard"], w, label="Standard",  color="#1565C0", alpha=0.85)
    axes[0,1].bar(xf,   top10_fc_rul["RUL_decay"],    w, label="Decay×0.85",color="#FF6F00", alpha=0.85)
    axes[0,1].bar(xf+w, top10_fc_rul["RUL_rolling"],  w, label="Rolling",   color="#2E7D32", alpha=0.85)
    axes[0,1].set_xticks(xf)
    axes[0,1].set_xticklabels([f"FC={fc}" for fc in top10_fc_rul[FC_COL]], rotation=45, fontsize=8)
    axes[0,1].set(title="RUL per Fault Code (top 10 by frequency)", ylabel="RUL (days)")
    axes[0,1].legend()

    # Rolling window RUL — top 30 devices
    axes[1,0].bar(x, top30_dev["RUL_rolling"].values, color="#2E7D32", alpha=0.85)
    axes[1,0].set_xticks(x)
    axes[1,0].set_xticklabels([str(d) for d in top30_dev[DEV_KEY]], rotation=90, fontsize=6)
    axes[1,0].set(title="Rolling Window RUL — Top 30 Devices", ylabel="RUL (days)")

    # Rolling window RUL — top 10 fault codes
    axes[1,1].bar(xf, top10_fc_rul["RUL_rolling"].values, color="#7B1FA2", alpha=0.85)
    axes[1,1].set_xticks(xf)
    axes[1,1].set_xticklabels([f"FC={fc}" for fc in top10_fc_rul[FC_COL]], rotation=45, fontsize=8)
    axes[1,1].set(title="Rolling Window RUL — Top 10 Fault Codes", ylabel="RUL (days)")

    plt.tight_layout()
    plt.show()


In [ ]:
# ── Cell 25 ──────────────────────────────────────────────────────────
# ── Export to HTML ─────────────────────────────────────────────────────────────
import subprocess, os, datetime
_nb  = "/home/sagemaker-user/PS5_TVM_Device_Reliability_Analysis.ipynb"
_ts  = datetime.datetime.now().strftime("%Y%m%d_%H%M")
_out = _nb.replace(".ipynb", f"_output_{_ts}.html")
if os.path.exists(_nb):
    r = subprocess.run(["jupyter", "nbconvert", "--to", "html", _nb, "--output", _out],
                       capture_output=True, text=True)
    print(f"[HTML] {_out}" if r.returncode==0 else r.stderr[-300:])
else:
    print(f"[HTML] Notebook path: adjust _nb to match your SageMaker path.")
